# 05 - Baseline: CRNN + CTC trained from scratch

First OCR baseline, trained only on our own data (no pretrained weights), before comparing the starter pack's pretrained approaches (VLM/PaddleOCR/Kraken). Architecture: CNN feature extractor + bidirectional LSTM + CTC loss/decoding (Shi, Bai & Yao, 2015 - see RESOURCES.md), the standard approach for variable-length line-level text recognition. Diagram of the architecture: https://claude.ai/code/artifact/c0349616-105a-467d-87c3-e30a3df58d52

Trained on the full `fit` split (3,280 rows, ~24s/epoch on this GPU), evaluated on a fixed 120-row sample of the `val` split - the same sample size later used to compare PaddleOCR/Kraken/VLM, so results are comparable across approaches.

In [ ]:
import sys
sys.path.append('..')

import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset

from src.config import RANDOM_STATE
from src.data import build_image_path, load_raw_data, split_train_val
from src.evaluate import combined_score

device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
train = load_raw_data('train')
fit_df, val_df = split_train_val(train)

# Fixed small validation sample - same size/seed will be reused to compare
# PaddleOCR/Kraken/VLM against this baseline on equal footing.
val_sample = val_df.sample(120, random_state=RANDOM_STATE).reset_index(drop=True)
len(fit_df), len(val_df), len(val_sample)

In [ ]:
# Character vocabulary from the fit split (index 0 reserved for the CTC blank token)
chars = sorted(set(''.join(fit_df['Target'].tolist())))
BLANK = 0
char_to_idx = {c: i + 1 for i, c in enumerate(chars)}
idx_to_char = {i + 1: c for i, c in enumerate(chars)}
NUM_CLASSES = len(chars) + 1
NUM_CLASSES

In [ ]:
TARGET_HEIGHT = 32
MAX_WIDTH = 1024


def load_and_resize(record_id):
    img = Image.open(build_image_path(record_id)).convert('L')
    w, h = img.size
    new_w = max(8, min(MAX_WIDTH, round(w * TARGET_HEIGHT / h)))
    img = img.resize((new_w, TARGET_HEIGHT), Image.BILINEAR)
    arr = (np.asarray(img, dtype=np.float32) / 255.0 - 0.5) / 0.5
    return torch.from_numpy(arr).unsqueeze(0)  # 1 x H x W

**CTC feasibility check.** CTC requires each sample's post-CNN time-step count `T` to be >= its target text length (the network needs at least one time step per character to emit it). With height=32 and the CNN's 4x width downsampling (below), 9 of 3,280 fit rows violate this - all the same kind of cramped multi-line/odd crops flagged during image EDA (`notebooks/02_eda_images.ipynb`), e.g. `PO7QQLWIFOT65BTz` and `EcxuqKeZl7OQexfB`. `nn.CTCLoss(zero_infinity=True)` zeroes the loss/gradient for these specific samples instead of producing NaN/inf, so they're silently skipped rather than corrupting the batch - acceptable at this rate (0.27%) for a baseline.

In [ ]:
class LineDataset(Dataset):
    def __init__(self, df):
        self.ids = df['ID'].tolist()
        self.targets = df['Target'].tolist()

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        img = load_and_resize(self.ids[i])
        text = self.targets[i]
        label = torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)
        return img, label, text


def collate_fn(batch):
    imgs, labels, texts = zip(*batch)
    widths = [im.shape[-1] for im in imgs]
    max_w = max(widths)
    padded = torch.zeros(len(imgs), 1, TARGET_HEIGHT, max_w)
    for i, im in enumerate(imgs):
        padded[i, :, :, : im.shape[-1]] = im
    # Two width-halving maxpools in the CNN (conv1, conv2) -> total width stride 4.
    input_lengths = torch.tensor([(w // 2) // 2 for w in widths], dtype=torch.long)
    target_lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)
    return padded, torch.cat(labels), input_lengths, target_lengths, texts

In [ ]:
class CRNN(nn.Module):
    """CNN + BiLSTM + CTC head (Shi, Bai & Yao 2015), see RESOURCES.md."""

    def __init__(self, num_classes):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, 1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d((2, 1)),
            nn.Conv2d(128, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d((2, 1)),
            nn.Conv2d(128, 256, (2, 1), 1, 0), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
        )
        self.rnn = nn.LSTM(256, 128, num_layers=2, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        feat = self.cnn(x)  # B x C x 1 x W'
        feat = feat.squeeze(2).permute(0, 2, 1)  # B x W' x C
        seq, _ = self.rnn(feat)
        return self.fc(seq).log_softmax(2)  # B x W' x num_classes


def greedy_decode(log_probs, input_lengths):
    """Argmax per time step, collapse repeats, drop blanks."""
    pred_idx = log_probs.argmax(2)
    texts = []
    for b in range(pred_idx.size(0)):
        seq = pred_idx[b, : input_lengths[b]].tolist()
        out, prev = [], None
        for idx in seq:
            if idx != prev and idx != BLANK:
                out.append(idx_to_char[idx])
            prev = idx
        texts.append(''.join(out))
    return texts

In [ ]:
fit_ds = LineDataset(fit_df)
fit_dl = DataLoader(fit_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)

val_ds = LineDataset(val_sample)
val_dl = DataLoader(val_ds, batch_size=32, shuffle=False, collate_fn=collate_fn)

model = CRNN(NUM_CLASSES).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
ctc_loss = nn.CTCLoss(blank=BLANK, zero_infinity=True)
EPOCHS = 30

In [ ]:
def evaluate(loader):
    model.eval()
    refs, hyps = [], []
    with torch.no_grad():
        for imgs, targets_cat, input_lengths, target_lengths, texts in loader:
            imgs = imgs.to(device)
            log_probs = model(imgs)
            preds = greedy_decode(log_probs.cpu(), input_lengths)
            refs.extend(texts)
            hyps.extend(preds)
    model.train()
    return combined_score(refs, hyps)


t0 = time.time()
history = []
for epoch in range(1, EPOCHS + 1):
    total_loss = 0.0
    for imgs, targets_cat, input_lengths, target_lengths, texts in fit_dl:
        imgs = imgs.to(device)
        optimizer.zero_grad()
        log_probs = model(imgs)
        loss = ctc_loss(log_probs.permute(1, 0, 2), targets_cat, input_lengths, target_lengths)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
    train_loss = total_loss / len(fit_ds)
    history.append({'epoch': epoch, 'train_loss': train_loss})
    if epoch % 5 == 0 or epoch == EPOCHS:
        score = evaluate(val_dl)
        history[-1].update(score)
        print(f"epoch {epoch:2d} | loss {train_loss:.3f} | val CER {score['cer']:.3f} "
              f"WER {score['wer']:.3f} final {score['final']:.3f} | {time.time()-t0:.0f}s")
    else:
        print(f"epoch {epoch:2d} | loss {train_loss:.3f} | {time.time()-t0:.0f}s")

In [ ]:
final_score = evaluate(val_dl)
final_score

In [ ]:
# A few qualitative examples
model.eval()
with torch.no_grad():
    imgs, targets_cat, input_lengths, target_lengths, texts = next(iter(val_dl))
    imgs = imgs.to(device)
    preds = greedy_decode(model(imgs).cpu(), input_lengths)
for pred, ref in list(zip(preds, texts))[:8]:
    print(f"pred: {pred!r}")
    print(f"ref:  {ref!r}")
    print()

In [ ]:
from src.config import MODELS_DIR

MODELS_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODELS_DIR / 'crnn_baseline.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'char_to_idx': char_to_idx,
    'idx_to_char': idx_to_char,
    'target_height': TARGET_HEIGHT,
    'max_width': MAX_WIDTH,
}, model_path)
model_path